# Salary Prediction Using Linear Regression
**IBM Applied AI & Machine Learning Internship Project**

This notebook walks through the complete ML pipeline:
CSV Dataset → Data Preprocessing → Train/Test Split → Linear Regression → Evaluation → Visualization → Prediction


## Step 0 – Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Libraries imported successfully!')

## Step 1 – Load the Dataset

In [ ]:
# Load the CSV dataset
df = pd.read_csv('../dataset/Salary_Data.csv')
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(10)

## Step 2 – Explore & Clean the Data

In [ ]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum())

# Basic statistics
print('\nBasic Statistics:')
df.describe()

In [ ]:
# Check data types
print('Data types:')
print(df.dtypes)

# Check for negative years
negative_years = df[df['YearsExperience'] < 0]
print(f'\nRows with negative YearsExperience: {len(negative_years)}')

## Step 3 – Define Features (X) and Target (y)

In [ ]:
# X = Independent variable (Feature)
# y = Dependent variable (Target)
X = df[['YearsExperience']].values  # 2D array for sklearn
y = df['Salary'].values

print(f'X shape: {X.shape}  (feature matrix)')
print(f'y shape: {y.shape}  (target vector)')

## Step 4 – Train / Test Split

In [ ]:
# Split: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

## Step 5 – Train the Linear Regression Model

In [ ]:
# Create and train the model
model = LinearRegression()
model.fit(X_train, y_train)

print('Model trained successfully!')
print(f'Coefficient  (slope)    : {model.coef_[0]:,.4f}')
print(f'Intercept               : {model.intercept_:,.4f}')
print(f'\nRegression Equation:')
print(f'  Salary = {model.intercept_:,.2f} + {model.coef_[0]:,.2f} × YearsExperience')

## Step 6 – Generate Predictions

In [ ]:
# Predict on the test set
y_pred = model.predict(X_test)

# Compare actual vs predicted
comparison = pd.DataFrame({
    'Years of Experience': X_test.flatten(),
    'Actual Salary'      : y_test,
    'Predicted Salary'   : y_pred.round(0)
})
comparison.sort_values('Years of Experience', inplace=True)
comparison.reset_index(drop=True, inplace=True)
comparison

## Step 7 – Evaluate the Model

In [ ]:
# Calculate evaluation metrics
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('=' * 40)
print('  MODEL EVALUATION METRICS')
print('=' * 40)
print(f'  R² Score : {r2:.4f}   (higher is better, max = 1)')
print(f'  MAE      : ₹{mae:,.2f}')
print(f'  MSE      : ₹{mse:,.2f}')
print(f'  RMSE     : ₹{rmse:,.2f}  (same unit as Salary)')
print('=' * 40)

## Step 8 – Visualize: Scatter Plot with Regression Line

In [ ]:
# Generate regression line points
x_min = df['YearsExperience'].min()
x_max = df['YearsExperience'].max()
x_line = np.linspace(x_min, x_max, 100).reshape(-1, 1)
y_line = model.predict(x_line)

fig = go.Figure()

# Actual data points
fig.add_trace(go.Scatter(
    x=df['YearsExperience'].tolist(),
    y=df['Salary'].tolist(),
    mode='markers',
    name='Actual Data',
    marker=dict(color='#3b82f6', size=10),
    hovertemplate='<b>Experience:</b> %{x} yrs<br><b>Salary:</b> ₹%{y:,.0f}<extra></extra>',
))

# Regression line
fig.add_trace(go.Scatter(
    x=x_line.flatten().tolist(),
    y=y_line.tolist(),
    mode='lines',
    name='Regression Line',
    line=dict(color='#ef4444', width=2.5),
))

fig.update_layout(
    title='Years of Experience vs Salary',
    xaxis_title='Years of Experience',
    yaxis_title='Salary (₹)',
    height=480,
)
fig.show()

## Step 9 – Visualize: Actual vs Predicted Salary

In [ ]:
y_all_pred = model.predict(X)
actual = y.tolist()
predicted = y_all_pred.tolist()
min_v, max_v = min(min(actual), min(predicted)), max(max(actual), max(predicted))

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=[min_v, max_v], y=[min_v, max_v],
    mode='lines', name='Perfect Prediction',
    line=dict(color='gray', dash='dash')))
fig2.add_trace(go.Scatter(x=actual, y=predicted,
    mode='markers', name='Predicted vs Actual',
    marker=dict(color='#8b5cf6', size=10),
    hovertemplate='Actual: ₹%{x:,.0f}<br>Predicted: ₹%{y:,.0f}<extra></extra>'))
fig2.update_layout(
    title='Actual Salary vs Predicted Salary',
    xaxis_title='Actual Salary (₹)',
    yaxis_title='Predicted Salary (₹)',
    height=480,
)
fig2.show()

## Step 10 – Predict Salary for New Input

In [ ]:
# Enter the years of experience you want to predict for
years_input = 5.5

predicted_salary = model.predict([[years_input]])[0]
print(f'Years of Experience  : {years_input}')
print(f'Predicted Salary     : ₹{predicted_salary:,.0f}')
print('(This is a model estimate, not a guaranteed salary)')